# Fusion: Text + WavLM

Combines the two approaches 11 different ways and compares them. Precision-first evaluation.

**Base models** (retrained from cached features):
1. Text-RF: RandomForest on stylometric+formal_ai+disfluency+pause features (~40 feats)
2. Text-XGB-top5: XGBoost on the 5 strongest features (mattr, mtld, avg_word_length, ttr, formal_transition_count)
3. WavLM Whole+Pretrained (768 dims)
4. WavLM Seg+Pretrained (1536 dims)

**Fusion methods:**
- Weighted average (alpha sweep) per pair + 4-way
- Vote-AND / Vote-OR / Majority
- Geometric mean / Rank fusion
- Meta-learner stacking (LogReg on OOF probas, XGBoost on OOF probas)
- Early fusion: concatenate text + wavlm features into single XGBoost
- Confidence gating

Relies on cached CSVs from v3 / wavlm_4way_comparison / text_cheating_detection notebooks.

In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================
from pathlib import Path

TRAIN_FOLDERS = ["audios2", "audios4"]
TEST_FOLDER   = "audios5"

# Text feature groups to include (skip low-signal groups found in ablation)
TEXT_GROUPS = ['stylometric', 'formal_ai', 'disfluency', 'pause']
TOP5_TEXT_FEATS = ['mattr', 'mtld', 'avg_word_length', 'ttr', 'formal_transition_count']

TEST_RATIO  = 0.20
RANDOM_SEED = 42

# ---------------- Whisper 3-way fusion switch ----------------
# 0 -> original 2-way fusion: 0.6*text_rf + 0.4*wavlm_wp (the 80P/70R audios5 result)
# 1 -> 3-way fusion: W_TEXT*text_rf + W_WAVLM*wavlm_wp + W_WHISPER*whisper_wp
USE_WHISPER = 0

# 3-way weights (ignored when USE_WHISPER=0). Must sum to 1.0.
W_TEXT    = 0.5
W_WAVLM   = 0.3
W_WHISPER = 0.2
# -------------------------------------------------------------

NB_DIR   = Path('.').resolve()
SAVE_DIR = NB_DIR / 'checkpoints_fusion'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}
print(f'Train: {TRAIN_FOLDERS}  |  Test: {TEST_FOLDER}')
print(f'USE_WHISPER = {USE_WHISPER}   (0 = 2-way baseline, 1 = 3-way with whisper)')
if USE_WHISPER:
    s = W_TEXT + W_WAVLM + W_WHISPER
    assert abs(s - 1.0) < 1e-6, f'weights must sum to 1.0, got {s}'
    print(f'  weights: text={W_TEXT}  wavlm={W_WAVLM}  whisper={W_WHISPER}')

In [ ]:
import json, warnings, itertools
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.stats import rankdata
import joblib
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, accuracy_score)
warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

# Feature groups (match text_cheating_detection notebook)
GROUPS = {
    'disfluency':  ['filler_rate','filler_count','repetition_rate','repair_rate',
                    'discourse_marker_rate','hedge_rate'],
    'stylometric': ['ttr','mattr','mtld','complex_word_rate','avg_word_length',
                    'n_words','n_unique_words','avg_sentence_length','std_sentence_length',
                    'fragment_rate','n_sentences','self_ref_rate',
                    'noun_rate','verb_rate','adj_rate'],
    'pause':       ['pause_mean','pause_std','pause_median','pause_skew','long_pause_rate',
                    'pause_ratio','n_pauses','pause_regularity',
                    'pause_before_content_ratio','pause_before_function_ratio',
                    'mid_phrase_pause_rate','words_per_sec','articulation_rate',
                    'initial_pause','longest_pause'],
    'formal_ai':   ['formal_transition_count','formal_transition_rate',
                    'ai_phrase_count','ai_phrase_rate'],
}
TEXT_FEATURES = [f for g in TEXT_GROUPS for f in GROUPS[g]]
print(f'Text features used: {len(TEXT_FEATURES)}')

## 1. Load Cached Features + Build Splits

In [ ]:
def load_gt(name):
    gt = pd.read_csv(NB_DIR / f'{name}GT.csv')
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col:'filename', lbl_col:'label_raw'})
    gt['label_int'] = gt['label_raw'].map(lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def load_folder(name):
    gt = load_gt(name)
    text_csv = NB_DIR / f'{name}_features.csv'
    wp_csv   = NB_DIR / f'{name}_whole_pretrained.csv'
    sp_csv   = NB_DIR / f'{name}_seg_pretrained.csv'
    for p in (text_csv, wp_csv, sp_csv):
        if not p.exists(): raise FileNotFoundError(p)
    text = pd.read_csv(text_csv)
    wp   = pd.read_csv(wp_csv)
    sp   = pd.read_csv(sp_csv)
    df = (gt.merge(text, on='filename', how='inner')
            .merge(wp,   on='filename', how='inner', suffixes=('','_wp'))
            .merge(sp,   on='filename', how='inner', suffixes=('','_sp')))
    # Optional whisper merge (only when switch is on)
    if USE_WHISPER:
        wh_csv = NB_DIR / f'{name}_whisper_whole.csv'
        if not wh_csv.exists():
            raise FileNotFoundError(f'{wh_csv} (USE_WHISPER=1 requires whisper cache)')
        wh = pd.read_csv(wh_csv)
        df = df.merge(wh, on='filename', how='inner', suffixes=('','_wh'))
    df['batch'] = name
    return df

train_df = pd.concat([load_folder(n) for n in TRAIN_FOLDERS], ignore_index=True)
test_df  = load_folder(TEST_FOLDER)

wp_cols = [c for c in train_df.columns if c.startswith('wavlm_') and not c.startswith('wavlm_mean_') and not c.startswith('wavlm_std_')]
sp_cols = [c for c in train_df.columns if c.startswith('wavlm_mean_') or c.startswith('wavlm_std_')]
text_cols = [c for c in TEXT_FEATURES if c in train_df.columns]
wh_cols = [c for c in train_df.columns if c.startswith('whisper_')] if USE_WHISPER else []

y_tr = train_df['label_int'].values
y_te = test_df['label_int'].values
filenames_te = test_df['filename'].values

X_text_tr = train_df[text_cols].fillna(0).values
X_text_te = test_df[text_cols].fillna(0).values
X_top5_tr = train_df[TOP5_TEXT_FEATS].fillna(0).values
X_top5_te = test_df[TOP5_TEXT_FEATS].fillna(0).values
X_wp_tr   = train_df[wp_cols].fillna(0).values
X_wp_te   = test_df[wp_cols].fillna(0).values
X_sp_tr   = train_df[sp_cols].fillna(0).values
X_sp_te   = test_df[sp_cols].fillna(0).values

if USE_WHISPER:
    X_wh_tr = train_df[wh_cols].fillna(0).values
    X_wh_te = test_df[wh_cols].fillna(0).values

print(f'Train: {len(train_df)}  (cheat={int((y_tr==1).sum())}, honest={int((y_tr==0).sum())})')
print(f'Test:  {len(test_df)}   (cheat={int((y_te==1).sum())}, honest={int((y_te==0).sum())})')
print(f'Text feats: {len(text_cols)}  | Top5: {len(TOP5_TEXT_FEATS)}  | WP: {len(wp_cols)}  | SP: {len(sp_cols)}' + (f'  | Whisper: {len(wh_cols)}' if USE_WHISPER else ''))
spw = (y_tr==0).sum() / max((y_tr==1).sum(), 1)
print(f'scale_pos_weight = {spw:.2f}')

## 2. Train Base Models (test probas + OOF train probas for stacking)

In [ ]:
def fit_xgb(n_feats):
    colsample = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
        scale_pos_weight=spw, eval_metric='logloss',
        random_state=RANDOM_SEED,
    )

def train_and_predict(make_clf, X_tr, X_te, y_tr):
    """Returns: test_proba (trained on full), oof_train_proba (5-fold CV)."""
    sc_full = StandardScaler().fit(X_tr)
    clf = make_clf()
    clf.fit(sc_full.transform(X_tr), y_tr)
    test_p = clf.predict_proba(sc_full.transform(X_te))[:, 1]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    oof = np.zeros(len(y_tr))
    for tr_i, va_i in skf.split(X_tr, y_tr):
        sc = StandardScaler().fit(X_tr[tr_i])
        m  = make_clf()
        m.fit(sc.transform(X_tr[tr_i]), y_tr[tr_i])
        oof[va_i] = m.predict_proba(sc.transform(X_tr[va_i]))[:, 1]
    return test_p, oof, clf, sc_full

make_text_rf   = lambda: RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_leaf=3,
                                                  class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)
make_top5_xgb  = lambda: fit_xgb(5)
make_wp_xgb    = lambda: fit_xgb(len(wp_cols))
make_sp_xgb    = lambda: fit_xgb(len(sp_cols))

print('Training Text-RF ...')
p_text, oof_text, _, _ = train_and_predict(make_text_rf,  X_text_tr, X_text_te, y_tr)
print('Training Text-Top5 XGB ...')
p_top5, oof_top5, _, _ = train_and_predict(make_top5_xgb, X_top5_tr, X_top5_te, y_tr)
print('Training WavLM Whole+Pre XGB ...')
p_wp, oof_wp, _, _     = train_and_predict(make_wp_xgb,   X_wp_tr,   X_wp_te,   y_tr)
print('Training WavLM Seg+Pre XGB ...')
p_sp, oof_sp, _, _     = train_and_predict(make_sp_xgb,   X_sp_tr,   X_sp_te,   y_tr)

BASE_TEST = {'text_rf': p_text, 'text_top5': p_top5, 'wavlm_wp': p_wp, 'wavlm_sp': p_sp}
BASE_OOF  = {'text_rf': oof_text, 'text_top5': oof_top5, 'wavlm_wp': oof_wp, 'wavlm_sp': oof_sp}

# Optional: Whisper base model (only when switch is on)
if USE_WHISPER:
    make_wh_xgb = lambda: fit_xgb(len(wh_cols))
    print('Training Whisper Whole+Pre XGB ...')
    p_wh, oof_wh, _, _ = train_and_predict(make_wh_xgb, X_wh_tr, X_wh_te, y_tr)
    BASE_TEST['whisper_wp'] = p_wh
    BASE_OOF['whisper_wp']  = oof_wh

print('\nBase model done.' + ('  [whisper included]' if USE_WHISPER else '  [2-way baseline]'))

## 3. Evaluation Utility

In [ ]:
def best_f1_metrics(proba, y, thr_grid=np.arange(0.20, 0.81, 0.02)):
    best_f1, best_thr = 0.0, 0.5
    for thr in thr_grid:
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    pred = (proba >= best_thr).astype(int)
    cm = confusion_matrix(y, pred, labels=[0,1])
    return dict(
        thr=round(float(best_thr),2),
        precision=round(precision_score(y, pred, zero_division=0),4),
        recall=round(recall_score(y, pred, zero_division=0),4),
        f1=round(best_f1,4),
        tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0]),
    )

def max_recall_at_precision(proba, y, target, thr_grid=np.arange(0.98, 0.20, -0.02)):
    best = None
    for thr in thr_grid:
        pred = (proba >= thr).astype(int)
        cm = confusion_matrix(y, pred, labels=[0,1])
        if cm[1,1] < 3: continue
        p = precision_score(y, pred, zero_division=0)
        r = recall_score(y, pred, zero_division=0)
        if p >= target and (best is None or r > best['rec']):
            best = dict(thr=round(float(thr),2), prec=round(p,4), rec=round(r,4),
                         tp=int(cm[1,1]), fp=int(cm[0,1]))
    return best

def evaluate_proba(proba, y, name):
    m = best_f1_metrics(proba, y)
    out = {'method': name, **m}
    for tgt in (0.80, 0.85, 0.90, 0.95):
        r = max_recall_at_precision(proba, y, tgt)
        out[f'rec@P{int(tgt*100)}'] = r['rec'] if r else None
        out[f'thr@P{int(tgt*100)}'] = r['thr'] if r else None
    return out

def evaluate_preds(preds, y, name):
    cm = confusion_matrix(y, preds, labels=[0,1])
    return {
        'method': name, 'thr': None,
        'precision': round(precision_score(y, preds, zero_division=0),4),
        'recall':    round(recall_score(y, preds, zero_division=0),4),
        'f1':        round(f1_score(y, preds, zero_division=0),4),
        'tp': int(cm[1,1]), 'fp': int(cm[0,1]), 'fn': int(cm[1,0]), 'tn': int(cm[0,0]),
        'rec@P80': None, 'rec@P85': None, 'rec@P90': None, 'rec@P95': None,
        'thr@P80': None, 'thr@P85': None, 'thr@P90': None, 'thr@P95': None,
    }

## 4. Baselines

In [ ]:
results = []
for name, p in BASE_TEST.items():
    results.append(evaluate_proba(p, y_te, f'base:{name}'))
base_df = pd.DataFrame(results)
print('Base models:')
print(base_df[['method','thr','precision','recall','f1','rec@P85','rec@P90']].to_string(index=False))

## 5. Fusion Method A -- Weighted Average (alpha sweep for each pair)

For each pair of base models: sweep weight alpha in {0, 0.1, ..., 1.0}. Pick best-F1.

In [ ]:
pairs = list(itertools.combinations(BASE_TEST.keys(), 2))
wa_pair_rows = []
for a, b in pairs:
    best = None
    for alpha in np.arange(0.0, 1.01, 0.1):
        fused = alpha*BASE_TEST[a] + (1-alpha)*BASE_TEST[b]
        m = best_f1_metrics(fused, y_te)
        if best is None or m['f1'] > best['f1']:
            best = {**m, 'alpha': round(float(alpha),2), 'pair': f'{a}+{b}', 'proba': fused}
    wa_pair_rows.append(best)
    r = evaluate_proba(best['proba'], y_te, f"wavg:{best['pair']}@a={best['alpha']}")
    results.append(r)

print('Weighted-average pairs (best-F1 alpha):')
for b in sorted(wa_pair_rows, key=lambda x: -x['f1']):
    print(f"  {b['pair']:<26} alpha={b['alpha']:<5} prec={b['precision']:.4f}  rec={b['recall']:.4f}  f1={b['f1']:.4f}")

# Equal weights 4-way
fused_all_eq = sum(BASE_TEST.values()) / len(BASE_TEST)
results.append(evaluate_proba(fused_all_eq, y_te, 'wavg:all_4_equal'))

# F1-optimized 4-way weights via grid
best_4w = None
step = 0.2
for w1 in np.arange(0, 1.01, step):
    for w2 in np.arange(0, 1.01-w1, step):
        for w3 in np.arange(0, 1.01-w1-w2, step):
            w4 = 1 - w1 - w2 - w3
            if w4 < -1e-6: continue
            weights = [w1, w2, w3, w4]
            fused = sum(w*p for w,p in zip(weights, BASE_TEST.values()))
            m = best_f1_metrics(fused, y_te)
            if best_4w is None or m['f1'] > best_4w['f1']:
                best_4w = {**m, 'weights': [round(float(x),2) for x in weights], 'proba': fused}
print(f"\n4-way optimized weights {best_4w['weights']}: prec={best_4w['precision']:.4f}  rec={best_4w['recall']:.4f}  f1={best_4w['f1']:.4f}")
results.append(evaluate_proba(best_4w['proba'], y_te, f"wavg:all_4_opt{best_4w['weights']}"))

## 6. Fusion Method B -- Voting (AND / OR / Majority)

Uses per-model thresholds that maximize each base model's F1.

In [ ]:
# Per-model best-F1 thresholds
base_thrs = {name: best_f1_metrics(p, y_te)['thr'] for name, p in BASE_TEST.items()}
print('Per-model best-F1 thresholds:', base_thrs)

def votes_at(thrs):
    preds = {name: (BASE_TEST[name] >= thrs[name]).astype(int) for name in BASE_TEST}
    return preds

preds = votes_at(base_thrs)
stack = np.column_stack(list(preds.values()))

# AND, OR, Majority across all 4
and_all = (stack.sum(axis=1) == 4).astype(int)
or_all  = (stack.sum(axis=1) >= 1).astype(int)
maj_all = (stack.sum(axis=1) >= 3).astype(int)

# Text AND WavLM (the two precision leaders)
p_text_or = np.maximum(preds['text_rf'], preds['text_top5'])
p_wav_or  = np.maximum(preds['wavlm_wp'], preds['wavlm_sp'])
and_text_wav = np.minimum(p_text_or, p_wav_or)

# Precision-first variant: require text_rf AND wavlm_wp at stricter thresholds
pr_text = (BASE_TEST['text_rf']  >= 0.60).astype(int)
pr_wav  = (BASE_TEST['wavlm_wp'] >= 0.55).astype(int)
and_strict = (pr_text & pr_wav).astype(int)

for name, preds_ in [('vote:AND_all', and_all), ('vote:OR_all', or_all),
                     ('vote:majority', maj_all),
                     ('vote:AND_text_wav', and_text_wav),
                     ('vote:AND_strict(text>=0.60 AND wp>=0.55)', and_strict)]:
    results.append(evaluate_preds(preds_, y_te, name))
    m = results[-1]
    print(f"  {name:<42} prec={m['precision']:.4f}  rec={m['recall']:.4f}  f1={m['f1']:.4f}  tp={m['tp']} fp={m['fp']} fn={m['fn']}")

## 7. Fusion Method C -- Geometric Mean / Rank Fusion

In [ ]:
# Geometric mean (biases toward low-confidence items, which is good for precision)
stack_p = np.column_stack([BASE_TEST[k] for k in BASE_TEST])
geo_all = np.prod(stack_p, axis=1) ** (1.0/stack_p.shape[1])
geo_pair_best = None
for a, b in pairs:
    g = np.sqrt(BASE_TEST[a] * BASE_TEST[b])
    m = best_f1_metrics(g, y_te)
    if geo_pair_best is None or m['f1'] > geo_pair_best['f1']:
        geo_pair_best = {**m, 'pair': f'{a}+{b}', 'proba': g}

# Rank fusion (distribution-agnostic; good when probas are miscalibrated)
ranks = np.column_stack([rankdata(BASE_TEST[k]) / len(y_te) for k in BASE_TEST])
rank_all = ranks.mean(axis=1)

results.append(evaluate_proba(geo_all, y_te, 'geo_mean:all_4'))
results.append(evaluate_proba(geo_pair_best['proba'], y_te, f"geo_mean:{geo_pair_best['pair']}"))
results.append(evaluate_proba(rank_all, y_te, 'rank_fusion:all_4'))
for name in ['geo_mean:all_4', f"geo_mean:{geo_pair_best['pair']}", 'rank_fusion:all_4']:
    r = next(x for x in results if x['method'] == name)
    print(f"  {name:<30} prec={r['precision']:.4f}  rec={r['recall']:.4f}  f1={r['f1']:.4f}")

## 8. Fusion Method D -- Meta-Learner Stacking

Train on OOF train probas (no leakage), predict with full-model test probas.

In [ ]:
X_meta_tr = np.column_stack([BASE_OOF[k]  for k in BASE_TEST])
X_meta_te = np.column_stack([BASE_TEST[k] for k in BASE_TEST])

# Meta-LogReg
meta_lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED)
meta_lr.fit(X_meta_tr, y_tr)
p_meta_lr = meta_lr.predict_proba(X_meta_te)[:, 1]
print(f'Meta-LogReg coefs: {dict(zip(BASE_TEST.keys(), np.round(meta_lr.coef_[0], 3)))}')

# Meta-XGBoost
meta_xgb = xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                               scale_pos_weight=spw, random_state=RANDOM_SEED)
meta_xgb.fit(X_meta_tr, y_tr)
p_meta_xgb = meta_xgb.predict_proba(X_meta_te)[:, 1]

results.append(evaluate_proba(p_meta_lr,  y_te, 'stack:meta_logreg'))
results.append(evaluate_proba(p_meta_xgb, y_te, 'stack:meta_xgb'))
for n in ('stack:meta_logreg', 'stack:meta_xgb'):
    r = next(x for x in results if x['method']==n)
    print(f"  {n:<22} prec={r['precision']:.4f}  rec={r['recall']:.4f}  f1={r['f1']:.4f}  rec@P85={r['rec@P85']}  rec@P90={r['rec@P90']}")

## 9. Fusion Method E -- Early Fusion (feature-level concatenation)

In [ ]:
def train_early(X_tr, X_te, tag):
    sc = StandardScaler().fit(X_tr)
    clf = fit_xgb(X_tr.shape[1])
    clf.fit(sc.transform(X_tr), y_tr)
    return clf.predict_proba(sc.transform(X_te))[:, 1]

X_ef1_tr = np.hstack([X_text_tr, X_wp_tr])
X_ef1_te = np.hstack([X_text_te, X_wp_te])
p_ef1 = train_early(X_ef1_tr, X_ef1_te, 'text+wp')

X_ef2_tr = np.hstack([X_text_tr, X_wp_tr, X_sp_tr])
X_ef2_te = np.hstack([X_text_te, X_wp_te, X_sp_te])
p_ef2 = train_early(X_ef2_tr, X_ef2_te, 'text+wp+sp')

X_ef3_tr = np.hstack([X_top5_tr, X_wp_tr])
X_ef3_te = np.hstack([X_top5_te, X_wp_te])
p_ef3 = train_early(X_ef3_tr, X_ef3_te, 'top5+wp')

results.append(evaluate_proba(p_ef1, y_te, 'early:text+wp'))
results.append(evaluate_proba(p_ef2, y_te, 'early:text+wp+sp'))
results.append(evaluate_proba(p_ef3, y_te, 'early:top5+wp'))
for n in ('early:text+wp', 'early:text+wp+sp', 'early:top5+wp'):
    r = next(x for x in results if x['method']==n)
    print(f"  {n:<22} prec={r['precision']:.4f}  rec={r['recall']:.4f}  f1={r['f1']:.4f}")

## 10. Fusion Method F -- Confidence Gating

If either model is very confident, trust it; else take the mean. Crude but sometimes effective.

In [ ]:
def conf_gate(p_a, p_b, low=0.25, high=0.75):
    out = 0.5 * (p_a + p_b)
    # trust high-confidence predictions from either side
    mask_a_high = p_a >= high; mask_a_low = p_a <= low
    mask_b_high = p_b >= high; mask_b_low = p_b <= low
    out = np.where(mask_a_high | mask_a_low, p_a, out)
    out = np.where(mask_b_high | mask_b_low, p_b, out)
    return out

p_gate = conf_gate(BASE_TEST['text_rf'], BASE_TEST['wavlm_wp'])
results.append(evaluate_proba(p_gate, y_te, 'gate:text_rf<>wavlm_wp'))
r = results[-1]
print(f"  gate:text_rf<>wavlm_wp  prec={r['precision']:.4f}  rec={r['recall']:.4f}  f1={r['f1']:.4f}")

## 11. Master Comparison Table

In [ ]:
res_df = pd.DataFrame(results)
res_df = res_df.sort_values('f1', ascending=False).reset_index(drop=True)

print('\n' + '='*96)
print('  ALL METHODS (sorted by F1)')
print('='*96)
show = res_df[['method','thr','precision','recall','f1','tp','fp','fn',
                'rec@P85','rec@P90','rec@P95']]
print(show.to_string(index=False))

print('\n' + '='*96)
print('  Best method for each precision target (sorted by recall)')
print('='*96)
for tgt in (0.80, 0.85, 0.90, 0.95):
    col = f'rec@P{int(tgt*100)}'
    thr_col = f'thr@P{int(tgt*100)}'
    sub = res_df[res_df[col].notna()][['method', col, thr_col]].copy()
    sub = sub.sort_values(col, ascending=False)
    print(f'\n  Precision >= {tgt:.2f}  (recall | threshold)')
    for _, row in sub.head(8).iterrows():
        print(f"    {row['method']:<48} rec={row[col]:.4f}  thr={row[thr_col]}")

## 12. Threshold Sweep for Top Methods

In [ ]:
# Collect probas for the top-5 best-F1 methods that produced probas (skip pure vote-methods)
proba_sources = {
    'base:text_rf':           BASE_TEST['text_rf'],
    'base:text_top5':         BASE_TEST['text_top5'],
    'base:wavlm_wp':          BASE_TEST['wavlm_wp'],
    'base:wavlm_sp':          BASE_TEST['wavlm_sp'],
    'wavg:all_4_equal':       fused_all_eq,
    f"wavg:all_4_opt{best_4w['weights']}": best_4w['proba'],
    'stack:meta_logreg':      p_meta_lr,
    'stack:meta_xgb':         p_meta_xgb,
    'early:text+wp':          p_ef1,
    'early:text+wp+sp':       p_ef2,
    'early:top5+wp':          p_ef3,
    'geo_mean:all_4':         geo_all,
    'rank_fusion:all_4':      rank_all,
    'gate:text_rf<>wavlm_wp': p_gate,
}
# add pair weighted-averages
for b in wa_pair_rows:
    proba_sources[f"wavg:{b['pair']}@a={b['alpha']}"] = b['proba']

# Pick top 5 by F1
top5_methods = res_df[res_df['method'].isin(proba_sources.keys())].head(5)['method'].tolist()
print(f'Top-5 methods by F1: {top5_methods}\n')

THR = np.arange(0.20, 0.81, 0.05)
for name in top5_methods:
    p = proba_sources[name]
    print('='*70)
    print(f'  {name}')
    print('='*70)
    rows = []
    for t in THR:
        pred = (p >= t).astype(int)
        cm = confusion_matrix(y_te, pred, labels=[0,1])
        rows.append({
            'thr':round(float(t),2),
            'prec':round(precision_score(y_te, pred, zero_division=0),4),
            'rec': round(recall_score(y_te, pred, zero_division=0),4),
            'f1':  round(f1_score(y_te, pred, zero_division=0),4),
            'tp':int(cm[1,1]),'fp':int(cm[0,1]),'fn':int(cm[1,0]),
        })
    print(pd.DataFrame(rows).to_string(index=False))
    print()

## 13. Error Overlap Analysis

If models make the same mistakes, fusion won't help. Low overlap (low Jaccard) = complementary models = fusion should work.

In [ ]:
# At best-F1 thresholds, who gets what wrong?
errors = {}
for name, p in BASE_TEST.items():
    m = best_f1_metrics(p, y_te)
    pred = (p >= m['thr']).astype(int)
    errors[name] = set(np.where(pred != y_te)[0])

names = list(errors.keys())
print('Error set sizes:', {n: len(errors[n]) for n in names})
print()
print('Jaccard overlap of error sets (1.0 = same mistakes, 0.0 = complementary):')
print(f'{"":<12}' + ''.join(f'{n:>12}' for n in names))
for a in names:
    row = [f'{a:<12}']
    for b in names:
        if a == b: row.append(f'{1.0:>12.3f}')
        else:
            inter = len(errors[a] & errors[b])
            union = len(errors[a] | errors[b])
            j = inter / max(union, 1)
            row.append(f'{j:>12.3f}')
    print(''.join(row))

# File-level analysis: who each base model thinks is cheating
per_file = pd.DataFrame({'filename': filenames_te, 'label_int': y_te})
for name, p in BASE_TEST.items():
    per_file[f'p_{name}'] = np.round(p, 4)
    m = best_f1_metrics(p, y_te)
    per_file[f'pred_{name}'] = (p >= m['thr']).astype(int)

n_models = len(BASE_TEST)
per_file['agreement_count'] = per_file[[f'pred_{n}' for n in BASE_TEST]].sum(axis=1)
per_file.to_csv(SAVE_DIR / 'per_file_probas.csv', index=False)
print(f"\nSaved per-file probability/prediction table: {SAVE_DIR / 'per_file_probas.csv'}")

## 14. Cross-batch Generalization Check

Train on one batch, test on another. If fusion helps across swapped splits, it generalizes. If only on this split, it's overfit.

In [ ]:
def swap_eval(train_name, test_name):
    tr = load_folder(train_name); te = load_folder(test_name)
    yt, ye = tr['label_int'].values, te['label_int'].values
    spw_l = (yt==0).sum() / max((yt==1).sum(), 1)
    def local_fit_xgb(n):
        cs = 0.3 if n > 500 else 0.8
        return xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                   subsample=0.8, colsample_bytree=cs, min_child_weight=3,
                                   scale_pos_weight=spw_l, eval_metric='logloss', random_state=RANDOM_SEED)
    splits = {
        'text_rf':   (tr[text_cols].fillna(0).values, te[text_cols].fillna(0).values, make_text_rf()),
        'text_top5': (tr[TOP5_TEXT_FEATS].fillna(0).values, te[TOP5_TEXT_FEATS].fillna(0).values, local_fit_xgb(5)),
        'wavlm_wp':  (tr[wp_cols].fillna(0).values, te[wp_cols].fillna(0).values, local_fit_xgb(len(wp_cols))),
        'wavlm_sp':  (tr[sp_cols].fillna(0).values, te[sp_cols].fillna(0).values, local_fit_xgb(len(sp_cols))),
    }
    probs = {}
    for n, (Xt, Xe, clf) in splits.items():
        sc = StandardScaler().fit(Xt)
        clf.fit(sc.transform(Xt), yt)
        probs[n] = clf.predict_proba(sc.transform(Xe))[:, 1]
    rows = []
    for n, p in probs.items():
        rows.append({'train':train_name,'test':test_name,'method':f'base:{n}', **best_f1_metrics(p, ye)})
    # best wavg pair (text_rf + wavlm_wp) quick sweep
    best = None
    for alpha in np.arange(0.0, 1.01, 0.1):
        f = alpha*probs['text_rf'] + (1-alpha)*probs['wavlm_wp']
        m = best_f1_metrics(f, ye)
        if best is None or m['f1'] > best['f1']: best = {**m, 'alpha':round(float(alpha),2)}
    rows.append({'train':train_name,'test':test_name,'method':f"wavg:text_rf+wavlm_wp@a={best['alpha']}", **{k:v for k,v in best.items() if k!='alpha'}})
    # equal wavg 4-way
    fall = sum(probs.values()) / len(probs)
    rows.append({'train':train_name,'test':test_name,'method':'wavg:all_4_equal', **best_f1_metrics(fall, ye)})
    return rows

swaps = []
for a, b in [('audios2','audios4'), ('audios4','audios2'), ('audios2','audios5'), ('audios4','audios5')]:
    try: swaps.extend(swap_eval(a, b))
    except FileNotFoundError: print(f'skip {a}->{b}')

if swaps:
    sdf = pd.DataFrame(swaps)
    print('Cross-batch generalization (best-F1 on each split):')
    print(sdf[['train','test','method','precision','recall','f1']].to_string(index=False))

## 15. Save Everything

In [ ]:
res_df.to_csv(SAVE_DIR / 'fusion_comparison.csv', index=False)
if swaps:
    pd.DataFrame(swaps).to_csv(SAVE_DIR / 'cross_batch.csv', index=False)

# Save winner probabilities for downstream use
winner = res_df.iloc[0]
if winner['method'] in proba_sources:
    pd.DataFrame({'filename': filenames_te, 'label_int': y_te,
                  'fused_proba': proba_sources[winner['method']]}).to_csv(
        SAVE_DIR / 'winner_test_proba.csv', index=False)

summary = {
    'train': TRAIN_FOLDERS, 'test': TEST_FOLDER,
    'n_text_feats': len(text_cols), 'n_wp': len(wp_cols), 'n_sp': len(sp_cols),
    'winner_by_f1': winner.to_dict(),
    'base_thresholds': base_thrs,
    '4way_opt_weights': best_4w['weights'],
}
with open(SAVE_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Saved to {SAVE_DIR}/')
print('\nWinner (by best-F1):')
print(json.dumps({k: (str(v) if not isinstance(v,(int,float,type(None))) else v)
                  for k,v in winner.to_dict().items()}, indent=2, default=str))

## 16. Audios5 Review -- Full Predictions + Copy Misclassified Files

Uses the production-recommended fusion `0.6*text_rf + 0.4*wavlm_wp` (best precision/recall balance, best cross-batch generalizer).
Outputs:
- `audios5_full_predictions.csv` -- every file in audios5, sorted by fusion score descending. Columns for two thresholds (balanced=0.46, high-prec=0.75).
- `review/audios5_misclassified.csv` -- only misclassified rows (FP + FN at balanced thr), with blank `correct_gt` and `notes` columns for you to fill in.
- `review/*.wav` -- actual audio files copied for listening.


In [ ]:
# ================================================================
# AUDIOS5 REVIEW: full predictions + copy misclassified files
# ================================================================
import shutil

# Original 2-way production fusion (preserved when USE_WHISPER=0)
FUSION_ALPHA       = 0.6        # 0.6*text_rf + 0.4*wavlm_wp (audios5: ~80P/70R)
THR_BALANCED       = 0.46       # best-F1 operating point
THR_HIGH_PREC      = 0.75       # zero-false-accuse operating point

p_text_rf  = BASE_TEST['text_rf']
p_wavlm_wp = BASE_TEST['wavlm_wp']
p_wavlm_sp = BASE_TEST['wavlm_sp']

if USE_WHISPER:
    p_whisper_wp = BASE_TEST['whisper_wp']
    fusion_score = W_TEXT * p_text_rf + W_WAVLM * p_wavlm_wp + W_WHISPER * p_whisper_wp
    fusion_tag   = f'3way(text={W_TEXT}, wavlm={W_WAVLM}, whisper={W_WHISPER})'
else:
    p_whisper_wp = None
    fusion_score = FUSION_ALPHA * p_text_rf + (1 - FUSION_ALPHA) * p_wavlm_wp
    fusion_tag   = f'2way(text={FUSION_ALPHA}, wavlm={1-FUSION_ALPHA})'

print(f'Production fusion: {fusion_tag}')

# High-precision reference (text + wavlm_sp) -- unchanged, for cross-reference
fusion_highprec = 0.8 * p_text_rf + 0.2 * p_wavlm_sp

# Build full predictions table
review_cols = {
    'filename':       filenames_te,
    'current_gt':     y_te.astype(int),
    'text_rf_prob':   np.round(p_text_rf, 4),
    'wavlm_wp_prob':  np.round(p_wavlm_wp, 4),
    'wavlm_sp_prob':  np.round(p_wavlm_sp, 4),
}
if USE_WHISPER:
    review_cols['whisper_wp_prob'] = np.round(p_whisper_wp, 4)
review_cols['fusion_score']     = np.round(fusion_score, 4)
review_cols['fusion_highprec']  = np.round(fusion_highprec, 4)
review_df = pd.DataFrame(review_cols)

# Threshold predictions
review_df['pred_balanced']  = (fusion_score    >= THR_BALANCED ).astype(int)
review_df['pred_highprec']  = (fusion_highprec >= THR_HIGH_PREC).astype(int)

# Quick summary of production fusion performance at the balanced threshold
from sklearn.metrics import precision_score as _ps, recall_score as _rs, f1_score as _fs, confusion_matrix as _cm
_pred = review_df['pred_balanced'].values
_cmat = _cm(y_te, _pred, labels=[0,1])
print(f'\n[{fusion_tag} @ thr={THR_BALANCED}]')
print(f'  precision={_ps(y_te,_pred,zero_division=0):.4f}  recall={_rs(y_te,_pred,zero_division=0):.4f}  f1={_fs(y_te,_pred,zero_division=0):.4f}')
print(f'  tp={int(_cmat[1,1])}  fp={int(_cmat[0,1])}  fn={int(_cmat[1,0])}  tn={int(_cmat[0,0])}')

# Error classification (on balanced threshold)
def err_type(row):
    if row['current_gt'] == row['pred_balanced']:
        score = row['fusion_score']
        if THR_BALANCED - 0.08 <= score <= THR_BALANCED + 0.08:
            return 'CORRECT_BORDERLINE'
        return 'CORRECT'
    if row['current_gt'] == 0 and row['pred_balanced'] == 1:
        return 'FP'
    return 'FN'

review_df['error_type'] = review_df.apply(err_type, axis=1)

# Sort by fusion score descending (most confident cheat -> most confident honest)
review_df = review_df.sort_values('fusion_score', ascending=False).reset_index(drop=True)
review_df['rank'] = review_df.index + 1

# Columns for user to fill in during review
review_df['correct_gt'] = ''
review_df['notes']      = ''

col_order = ['rank', 'filename', 'current_gt', 'fusion_score', 'pred_balanced',
             'error_type', 'text_rf_prob', 'wavlm_wp_prob', 'wavlm_sp_prob']
if USE_WHISPER:
    col_order.append('whisper_wp_prob')
col_order += ['fusion_highprec', 'pred_highprec', 'correct_gt', 'notes']
review_df = review_df[col_order]

# Save full predictions
full_pred_suffix = '_whisper' if USE_WHISPER else ''
full_pred_path = SAVE_DIR / f'audios5_full_predictions{full_pred_suffix}.csv'
review_df.to_csv(full_pred_path, index=False)
print(f'\nFull predictions saved -> {full_pred_path}')

# ----------- Build review folder with misclassified files -----------
REVIEW_DIR = NB_DIR / f'review_audios5{full_pred_suffix}'
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

misc = review_df[review_df['error_type'].isin(['FP','FN','CORRECT_BORDERLINE'])].copy()
misc = misc.sort_values(['error_type', 'fusion_score'], ascending=[True, False]).reset_index(drop=True)
misc_csv = REVIEW_DIR / 'audios5_misclassified.csv'
misc.to_csv(misc_csv, index=False)

print(f'\nMisclassification summary (at thr={THR_BALANCED}):')
print(review_df['error_type'].value_counts().to_string())
print(f'\nReview CSV saved -> {misc_csv}')

# Copy misclassified audio files
src_candidates = [
    NB_DIR / TEST_FOLDER,
    NB_DIR.parent / TEST_FOLDER,
    NB_DIR / '..' / TEST_FOLDER,
]
src_dir = next((p for p in src_candidates if p.exists()), None)
if src_dir is None:
    print(f'\n[WARN] Could not find {TEST_FOLDER}/ folder. Tried: {[str(p) for p in src_candidates]}')
    print('Copy the audios manually from your audios5 folder into', REVIEW_DIR)
else:
    print(f'\nCopying {len(misc)} files from {src_dir} ...')
    copied, missing = 0, []
    for _, row in misc.iterrows():
        fn = row['filename']
        sub = REVIEW_DIR / row['error_type']
        sub.mkdir(parents=True, exist_ok=True)
        src_file = src_dir / fn
        if not src_file.exists():
            for ext in ('.wav', '.mp3', '.m4a', '.flac', '.ogg'):
                alt = src_dir / (fn + ext)
                if alt.exists():
                    src_file = alt
                    break
        if src_file.exists():
            rank = int(row['rank'])
            score = row['fusion_score']
            dest_name = f"r{rank:03d}_s{score:.3f}_gt{int(row['current_gt'])}_{src_file.name}"
            shutil.copy2(src_file, sub / dest_name)
            copied += 1
        else:
            missing.append(fn)
    print(f'Copied: {copied}  |  Missing: {len(missing)}')
    if missing:
        print('First 5 missing:', missing[:5])

print('\n' + '='*70)
print('REVIEW WORKFLOW:')
print('='*70)
print(f'1. Open: {misc_csv}')
print(f'2. Listen to files in {REVIEW_DIR.name}/FP/ and {REVIEW_DIR.name}/FN/')
print("3. Fill 'correct_gt' column (0/1) only if the current GT label is wrong")
print("4. Add any notes in the 'notes' column")
print('5. After review, rerun training with the corrected labels')
print('\nError types:')
print('  FP                 = model predicts cheat, GT says honest')
print('  FN                 = model predicts honest, GT says cheat')
print('  CORRECT_BORDERLINE = prediction matches GT but score is near threshold')